# 11.9 · 文本相似度 / Text Similarity

> **课程定位 / Where this fits**
> 第 9 课，**Part 11 · 经典 NLP**（本部分收尾）。
> Lesson 9, **Part 11 · Classic NLP** (finale of this part).
>
> "这两段文本有多像？"——这个看似简单的问题是**搜索引擎、问答匹配、去重、推荐、抄袭检测**的核心。但"相似"有多个层面：**字符级**(拼写/typo)、**集合级**(共享了哪些词)、**向量级**(语义)。本课**从零实现**四种最重要的相似度：**编辑距离、Jaccard、余弦相似度、以及搜索引擎的排序函数 BM25**，并演示一个**检索**例子。
> "How similar are these two texts?" — a deceptively simple question at the heart of **search engines, QA matching, deduplication, recommendation, plagiarism detection**. "Similar" has levels: **character-level** (spelling/typos), **set-level** (shared words), **vector-level** (semantics). We **implement from scratch** the four most important measures: **edit distance, Jaccard, cosine similarity, and BM25** (the search-engine ranking function), with a **retrieval** demo.
>
> 💼 **实战/面试视角**："编辑距离 DP / 余弦 vs Jaccard / BM25 为什么比 TF-IDF 好 / 语义相似度" 检索/搜索岗常考。
> 💼 **Practical/interview angle:** "edit-distance DP / cosine vs Jaccard / why BM25 > TF-IDF / semantic similarity" — search/IR questions.

> 📐 **符号约定 / Notation**
> - 编辑距离 —— 把一个串改成另一个串的最少操作数 / min edits to turn one string into another
> - 余弦相似度 —— 两向量夹角余弦 $\frac{a\cdot b}{\|a\|\|b\|}$ / cosine of the angle between vectors
> - BM25 —— 检索排序函数(TF-IDF 的改进) / a retrieval ranking function

> 💡 **面试相关 / Interview-relevant**
> - "编辑距离怎么用 DP 算"（出镜率 ★★★★）
> - "余弦相似度 vs 欧氏距离 vs Jaccard"（★★★★★）
> - "BM25 相比 TF-IDF 余弦的改进"（★★★★，词频饱和+长度归一化）
> - "字符级 vs 语义级相似度"（★★★）

---

## 学习目标 / Learning Objectives
1. 区分字符级/集合级/向量级相似度及适用场景。
   Distinguish character/set/vector similarity and their uses.
2. **从零实现编辑距离(DP)** 并可视化。
   Implement edit distance (DP) from scratch and visualize.
3. 实现 **Jaccard** 与 **余弦相似度**。
   Implement Jaccard and cosine similarity.
4. **从零实现 BM25** 并做检索演示。
   Implement BM25 from scratch and demo retrieval.

## 目录 / TOC
1. [相似度的三个层面 ⭐](#1)
2. [编辑距离（DP，从零）⭐](#2)
3. [Jaccard 与余弦相似度 ⭐](#3)
4. [BM25 检索（从零）+ 小结 ⭐](#4)


<a id="1"></a>
## 1. 相似度的三个层面 ⭐ / Three Levels of Similarity

"相似"取决于你关心什么：
"Similar" depends on what you care about:
- **字符级(character)**：`"color"` vs `"colour"`、拼写纠错、模糊匹配 → **编辑距离**。看字符怎么改。
  **Character-level:** `"color"` vs `"colour"`, spell-check, fuzzy match → **edit distance**. How characters differ.
- **集合级(set)**：两段文本**共享了哪些词** → **Jaccard**。快、适合去重，但忽略词频和顺序。
  **Set-level:** which words two texts **share** → **Jaccard**. Fast, good for dedup, ignores frequency/order.
- **向量级(vector)**：把文本变成向量(TF-IDF/词向量)再比 → **余弦相似度 / BM25**。能反映"主题/语义"相似，是**检索**的主力。
  **Vector-level:** turn texts into vectors (TF-IDF/embeddings) then compare → **cosine / BM25**. Captures topical/semantic similarity; the workhorse of **retrieval**.

没有"最好"的相似度，**看任务选**。下面逐一从零实现。
There's no single "best" — **choose by task**. We implement each from scratch.


<a id="2"></a>
## 2. 编辑距离（DP，从零）⭐ / Edit Distance (Levenshtein, DP)

**编辑距离(Levenshtein distance)**：把字符串 A 改成 B，最少需要多少次**单字符操作**(插入/删除/替换)。`"kitten"→"sitting"` 需要 3 步(k→s, e→i, 末尾加 g)。
**Edit distance (Levenshtein):** the minimum number of single-character **operations** (insert/delete/substitute) to turn string A into B. `"kitten"→"sitting"` takes 3 (k→s, e→i, append g).

用**动态规划**：`dp[i][j]` = A 的前 $i$ 个字符变成 B 的前 $j$ 个字符的最小操作数。递推：若 `A[i]==B[j]` 则不用动(取左上角)，否则取"插入/删除/替换"三者最小 + 1。这是 DP 的经典入门题(面试常考)。
Use **dynamic programming**: `dp[i][j]` = min edits to turn A's first $i$ chars into B's first $j$. Recurrence: if `A[i]==B[j]`, take the diagonal; else 1 + min(insert/delete/substitute). A classic DP interview problem.


In [ ]:
import numpy as np, matplotlib.pyplot as plt, seaborn as sns
sns.set_theme(style="white")

def edit_distance(a, b):
    m, n = len(a), len(b)
    dp = np.zeros((m+1, n+1), dtype=int)
    dp[0] = np.arange(n+1); dp[:, 0] = np.arange(m+1)    # 边界: 空串↔前缀=逐个插入/删除 / boundaries
    for i in range(1, m+1):
        for j in range(1, n+1):
            if a[i-1] == b[j-1]:
                dp[i][j] = dp[i-1][j-1]                   # 字符相同, 无需操作(取左上) / same char → diagonal
            else:
                dp[i][j] = 1 + min(dp[i-1][j],            # 删除 / delete
                                   dp[i][j-1],            # 插入 / insert
                                   dp[i-1][j-1])          # 替换 / substitute
    return dp

a, b = "kitten", "sitting"
dp = edit_distance(a, b)
print(f"'{a}' → '{b}' 的编辑距离 = {dp[-1][-1]}")
fig, ax = plt.subplots(figsize=(7, 5))
sns.heatmap(dp, annot=True, fmt="d", cmap="YlGnBu", cbar=False,
            xticklabels=["∅"]+list(b), yticklabels=["∅"]+list(a), ax=ax)
ax.set_title(f"编辑距离 DP 表: 右下角={dp[-1][-1]} 是答案\n每格=把A前i字符变成B前j字符的最少操作数")
ax.set_xlabel("目标串 B"); ax.set_ylabel("源串 A"); plt.tight_layout(); plt.show()
print("用途: 拼写纠错(找编辑距离最小的词典词)/模糊匹配/DNA比对; 复杂度 O(mn)")


<a id="3"></a>
## 3. Jaccard 与余弦相似度 ⭐ / Jaccard & Cosine Similarity

**Jaccard 相似度**(集合级)：两段文本的**词集合**的**交集 / 并集**。简单快速，衡量"共享了多少不同的词"，但**忽略词频和顺序**。常用于快速去重、集合重叠。
**Jaccard similarity** (set-level): **intersection / union** of the two texts' **word sets**. Simple and fast; measures "how many distinct words are shared," but **ignores frequency and order**. Used for quick dedup/overlap.

**余弦相似度**(向量级)：把文本变成向量(这里用 TF-IDF)，算两向量的**夹角余弦**。值 0~1，越大越相似。**只看方向不看长度**——所以长短不同但主题相同的文档也能判为相似(这正是它优于欧氏距离做文本相似的原因，呼应 Part 6)。
**Cosine similarity** (vector-level): vectorize texts (TF-IDF here) and take the **cosine of the angle**. 0–1, higher = more similar. **Direction, not magnitude** — so documents of different lengths but the same topic still score high (why cosine beats Euclidean for text, echoing Part 6).


In [ ]:
def jaccard(a, b):
    sa, sb = set(a.lower().split()), set(b.lower().split())   # 词集合 / word sets
    return len(sa & sb) / len(sa | sb)                        # 交集/并集 / intersection over union

from sklearn.feature_extraction.text import TfidfVectorizer
def cosine_pair(texts):
    X = TfidfVectorizer().fit_transform(texts)                # TF-IDF 向量 / TF-IDF vectors
    X = X.toarray(); X = X / (np.linalg.norm(X, axis=1, keepdims=True)+1e-9)   # L2 归一化 / normalize
    return X @ X.T                                            # 两两余弦相似度 / pairwise cosine

sents = [
    "the cat sat on the mat",
    "a cat is sitting on a mat",          # 与第1句语义近(用词不同) / similar meaning, different words
    "the dog ran in the park",            # 不同主题 / different topic
    "the cat sat on the mat",             # 与第1句完全相同 / identical to #0
]
print("Jaccard(集合重叠):")
print(f"  句0 vs 句1(语义近): {jaccard(sents[0],sents[1]):.2f}  (用词不同→Jaccard 偏低)")
print(f"  句0 vs 句3(完全相同): {jaccard(sents[0],sents[3]):.2f}")
print(f"  句0 vs 句2(不同主题): {jaccard(sents[0],sents[2]):.2f}")
cos = cosine_pair(sents)
fig, ax = plt.subplots(figsize=(5.5,4.5))
sns.heatmap(cos, annot=True, fmt=".2f", cmap="Reds", xticklabels=[f"句{i}" for i in range(4)],
            yticklabels=[f"句{i}" for i in range(4)], ax=ax)
ax.set_title("余弦相似度矩阵(TF-IDF): 对角=1, 相同句=1, 同主题较高"); plt.tight_layout(); plt.show()
print("\nJaccard: 快/看集合重叠/忽略词频; 余弦: 看向量方向, 长度无关, 适合文档相似/检索")


<a id="4"></a>
## 4. BM25 检索（从零）+ 小结 ⭐ / BM25 Retrieval From Scratch

**BM25** 是**搜索引擎排序的事实标准**(Elasticsearch/Lucene 默认)。给定查询，给每篇文档打分、排序。它是 TF-IDF 的改进，关键两点(面试重点)：
**BM25** is the **de facto search-engine ranking function** (default in Elasticsearch/Lucene). Given a query, it scores and ranks documents. It improves on TF-IDF in two key ways (interview):
1. **词频饱和(saturation)**：一个词在文档里出现 100 次 ≠ 比出现 10 次重要 10 倍。BM25 让 TF 的贡献**逐渐饱和**(由参数 $k_1$ 控制)，避免堆词刷分。
   **Term-frequency saturation:** 100 occurrences ≠ 10× more relevant than 10. BM25 makes TF contribution **saturate** (parameter $k_1$), preventing keyword stuffing.
2. **文档长度归一化**：长文档天然含更多词，不该因此占便宜。BM25 按文档长度相对平均长度做**惩罚**(参数 $b$)。
   **Document-length normalization:** long docs naturally contain more words; BM25 **penalizes** length relative to average (parameter $b$).

公式(对查询里每个词求和)：
Formula (sum over query terms):
$$\text{score}(D,Q)=\sum_{t\in Q}\text{IDF}(t)\cdot\frac{f(t,D)\,(k_1+1)}{f(t,D)+k_1\,(1-b+b\frac{|D|}{\text{avgdl}})}$$

下面**从零实现 BM25**，在一个小语料上跑检索。
We **implement BM25 from scratch** and run retrieval on a small corpus.


In [ ]:
import re
from collections import Counter
from sklearn.datasets import fetch_20newsgroups

# 小语料作为"被检索的文档库" / a small corpus to search over
raw = fetch_20newsgroups(subset="train", categories=["sci.space","rec.sport.baseball","comp.graphics"],
                         remove=("headers","footers","quotes")).data[:300]
def tok(s): return re.findall(r"[a-z]+", s.lower())
corpus = [tok(d) for d in raw]
N = len(corpus); avgdl = np.mean([len(d) for d in corpus])
df = Counter()                                            # 文档频率 / document frequency
for d in corpus:
    for w in set(d): df[w] += 1

def bm25_score(query, doc, k1=1.5, b=0.75):
    score = 0.0; dl = len(doc); tf = Counter(doc)
    for t in set(tok(query)):
        if t not in df: continue
        idf = np.log((N - df[t] + 0.5) / (df[t] + 0.5) + 1)          # BM25 的 IDF / BM25 IDF
        freq = tf[t]
        denom = freq + k1 * (1 - b + b * dl / avgdl)                 # 长度归一化 / length normalization
        score += idf * (freq * (k1 + 1)) / denom                     # 词频饱和 / TF saturation
    return score

def search(query, k=3):
    scores = [(i, bm25_score(query, corpus[i])) for i in range(N)]
    return sorted(scores, key=lambda x: -x[1])[:k]                   # 按分数排序取 top-k / rank by score

query = "rocket launch into orbit"
print(f"查询: '{query}'\nBM25 检索到的最相关文档:")
for rank, (i, sc) in enumerate(search(query), 1):
    print(f"  #{rank} (BM25={sc:.2f}): {' '.join(corpus[i][:18])}...")
print("\nBM25 把含 rocket/launch/orbit 且不靠长度灌水的文档排在前面 → 命中'太空'主题")


In [ ]:
# 演示词频饱和: 出现次数增加, BM25 贡献趋于饱和(非线性) / TF saturation curve
k1 = 1.5
freqs = np.arange(0, 21)
bm25_tf = (freqs * (k1+1)) / (freqs + k1)                 # 忽略长度项的 TF 贡献 / TF contribution
fig, ax = plt.subplots(figsize=(7,3.8))
ax.plot(freqs, freqs, "--", color="#bbb", label="纯词频(TF-IDF, 线性)")
ax.plot(freqs, bm25_tf, "o-", color="#39c", label=f"BM25 词频贡献(k1={k1}, 饱和)")
ax.set_xlabel("词在文档中出现次数"); ax.set_ylabel("对得分的贡献"); ax.legend()
ax.set_title("BM25 词频饱和: 出现越多边际贡献越小(防止堆词刷分)")
plt.tight_layout(); plt.show()
print("纯TF-IDF: 词频翻倍贡献翻倍(线性, 易被堆词操纵); BM25: 贡献饱和, 更稳健")
print("BM25 = 改进版TF-IDF: 词频饱和(k1) + 文档长度归一化(b); 检索排序的工业标准")


```
相似度三层面: 字符级(编辑距离,拼写/模糊) / 集合级(Jaccard,去重) / 向量级(余弦,BM25,检索)
编辑距离: DP, dp[i][j]=前缀最少操作(插/删/替); 相同取左上, 否则1+min三邻; O(mn)
Jaccard: 词集合 交集/并集; 快但忽略词频和顺序
余弦相似度: TF-IDF向量夹角余弦; 只看方向(长度无关) → 适合文档相似/检索(优于欧氏)
BM25: 搜索引擎排序标准; 改进TF-IDF两点: ①词频饱和(k1,防堆词) ②文档长度归一化(b)
没有最好的相似度, 看任务选; 现代语义检索用句向量(Sentence-BERT)+向量数据库
```

### 💡 面试速查 / Interview cheat-sheet
1. **编辑距离**: DP O(mn); 相同取左上, 否则 1+min(插/删/替)。
   Edit distance: DP O(mn); diagonal if equal, else 1+min(ins/del/sub).
2. **余弦 vs 欧氏 vs Jaccard**: 余弦看方向(长度无关), Jaccard看集合重叠。
   Cosine vs Euclidean vs Jaccard: cosine=direction (length-free), Jaccard=set overlap.
3. **BM25 > TF-IDF**: 词频饱和(k1) + 文档长度归一化(b)。
   BM25 > TF-IDF: TF saturation (k1) + length normalization (b).
4. **选择**: 拼写→编辑距离; 去重→Jaccard; 检索→BM25/余弦。
   Choose: spelling→edit distance; dedup→Jaccard; retrieval→BM25/cosine.
5. **语义相似**: 词/句向量(cosine)解决"用词不同但意思相近"。
   Semantic similarity: word/sentence embeddings (cosine) for synonymous phrasing.

### 🎉 Part 11 完成 / Part 11 Complete
你已走完**经典 NLP**：预处理、BoW/TF-IDF、词向量(Word2Vec 从零)、文本分类、情感分析、主题模型、NER、序列标注(HMM+Viterbi 从零)、文本相似度(BM25 从零)。这些是深度学习之前的 NLP 主干，至今仍是强基线与面试重点。下一站 **Part 12 现代 NLP 与大模型(Transformer/BERT/GPT)** 将把这些带入深度学习时代。
You've completed **Classic NLP**: preprocessing, BoW/TF-IDF, word embeddings (Word2Vec from scratch), classification, sentiment, topic models, NER, sequence labeling (HMM+Viterbi from scratch), and text similarity (BM25 from scratch). The pre-deep-learning NLP backbone — still strong baselines and interview staples. Next, **Part 12 Modern NLP & LLMs (Transformer/BERT/GPT)** brings these into the deep-learning era.
